# W_DM_ACTIVITY_TYPE_Full_Load.sql Conversion

**Converted to Databricks Spark SQL Jupyter Notebook**

**Conversion Timestamp:** 2024-05-20T12:00:00Z

**Description:** This notebook performs a full load of activity type dimension data into `workspace.target_schema.w_dm_activity_type`.
It stages data from `workspace.source_schema.c_w_dm_activity_type`, generates surrogate keys, and merges changes into the target table.

In [ ]:
dbutils.widgets.text("ETL_PROC_WID", "")

## ETL Parameters

In [ ]:
%sql
CREATE OR REPLACE TEMPORARY VIEW v_etl_parameters AS
SELECT
  CAST('${ETL_PROC_WID}' AS BIGINT) AS etl_proc_wid;

In [ ]:
display(spark.sql("SELECT * FROM v_etl_parameters;"))

## Staging Table

In [ ]:
%sql
-- SCEN_TASK_NO {1}: Drop staging table
DROP TABLE IF EXISTS workspace.source_schema.c_w_dm_activity_type_stg;

In [ ]:
%sql
-- SCEN_TASK_NO {2}: Create staging table
CREATE TABLE workspace.source_schema.c_w_dm_activity_type_stg
(
  ACTIVITY_TYPE_SK  BIGINT,
  ACTIVITY_TYPE_ID  STRING,
  ACTIVITY_TYPE_NAME  STRING,
  ACTIVITY_TYPE_DESC  STRING,
  ACTIVE_FLG  STRING,
  START_DATE  TIMESTAMP,
  END_DATE  TIMESTAMP,
  DELETE_FLG  STRING,
  INTEGRATION_ID  STRING,
  DATASOURCE_NUM_ID  BIGINT,
  ETL_INS_AUD_ID  BIGINT,
  ETL_UPD_AUD_ID  BIGINT,
  LOAD_DT  TIMESTAMP,
  IND_UPDATE  STRING,
  ODI_ROW_ID  STRING
)
USING DELTA;

In [ ]:
%sql
-- SCEN_TASK_NO {3}: Insert into staging table
INSERT INTO workspace.source_schema.c_w_dm_activity_type_stg
(
  ACTIVITY_TYPE_SK,
  ACTIVITY_TYPE_ID,
  ACTIVITY_TYPE_NAME,
  ACTIVITY_TYPE_DESC,
  ACTIVE_FLG,
  START_DATE,
  END_DATE,
  DELETE_FLG,
  INTEGRATION_ID,
  DATASOURCE_NUM_ID,
  ETL_INS_AUD_ID,
  ETL_UPD_AUD_ID,
  LOAD_DT,
  IND_UPDATE,
  ODI_ROW_ID
)
SELECT
  -- Replaced ODI_SEQ_W_DM_ACTIVITY_TYPE.NEXTVAL with a deterministic hash of INTEGRATION_ID for surrogate key
  ABS(xxhash64(C_DM_ACTIVITY_TYPE.INTEGRATION_ID)),
  C_DM_ACTIVITY_TYPE.ACTIVITY_TYPE_ID,
  C_DM_ACTIVITY_TYPE.ACTIVITY_TYPE_NAME,
  C_DM_ACTIVITY_TYPE.ACTIVITY_TYPE_DESC,
  C_DM_ACTIVITY_TYPE.ACTIVE_FLG,
  C_DM_ACTIVITY_TYPE.START_DATE,
  C_DM_ACTIVITY_TYPE.END_DATE,
  C_DM_ACTIVITY_TYPE.DELETE_FLG,
  C_DM_ACTIVITY_TYPE.INTEGRATION_ID,
  C_DM_ACTIVITY_TYPE.DATASOURCE_NUM_ID,
  (SELECT etl_proc_wid FROM v_etl_parameters),
  (SELECT etl_proc_wid FROM v_etl_parameters),
  current_timestamp(), -- Replaced SYSDATE
  'I',
  uuid() -- Replaced SYS_GUID()
FROM
  workspace.source_schema.c_w_dm_activity_type AS C_DM_ACTIVITY_TYPE;

In [ ]:
%sql
SELECT COUNT(*) AS record_count FROM workspace.source_schema.c_w_dm_activity_type_stg;

## MERGE into Target

In [ ]:
%sql
-- SCEN_TASK_NO {4}: MERGE into W_DM_ACTIVITY_TYPE
MERGE INTO workspace.target_schema.w_dm_activity_type AS T
USING workspace.source_schema.c_w_dm_activity_type_stg AS S
ON
  (
    T.INTEGRATION_ID = S.INTEGRATION_ID
  )
WHEN MATCHED THEN UPDATE SET
  T.ACTIVITY_TYPE_SK = S.ACTIVITY_TYPE_SK,
  T.ACTIVITY_TYPE_ID = S.ACTIVITY_TYPE_ID,
  T.ACTIVITY_TYPE_NAME = S.ACTIVITY_TYPE_NAME,
  T.ACTIVITY_TYPE_DESC = S.ACTIVITY_TYPE_DESC,
  T.ACTIVE_FLG = S.ACTIVE_FLG,
  T.START_DATE = S.START_DATE,
  T.END_DATE = S.END_DATE,
  T.DELETE_FLG = S.DELETE_FLG,
  T.DATASOURCE_NUM_ID = S.DATASOURCE_NUM_ID,
  T.ETL_UPD_AUD_ID = S.ETL_UPD_AUD_ID,
  T.LOAD_DT = S.LOAD_DT
WHEN NOT MATCHED THEN INSERT
(
  ACTIVITY_TYPE_SK,
  ACTIVITY_TYPE_ID,
  ACTIVITY_TYPE_NAME,
  ACTIVITY_TYPE_DESC,
  ACTIVE_FLG,
  START_DATE,
  END_DATE,
  DELETE_FLG,
  INTEGRATION_ID,
  DATASOURCE_NUM_ID,
  ETL_INS_AUD_ID,
  ETL_UPD_AUD_ID,
  LOAD_DT
)
VALUES
(
  S.ACTIVITY_TYPE_SK,
  S.ACTIVITY_TYPE_ID,
  S.ACTIVITY_TYPE_NAME,
  S.ACTIVITY_TYPE_DESC,
  S.ACTIVE_FLG,
  S.START_DATE,
  S.END_DATE,
  S.DELETE_FLG,
  S.INTEGRATION_ID,
  S.DATASOURCE_NUM_ID,
  S.ETL_INS_AUD_ID,
  S.ETL_UPD_AUD_ID,
  S.LOAD_DT
);

## Cleanup

In [ ]:
%sql
-- SCEN_TASK_NO {5}: Drop staging table
DROP TABLE IF EXISTS workspace.source_schema.c_w_dm_activity_type_stg;

## Validation

In [ ]:
%sql
SELECT COUNT(*) AS final_target_record_count FROM workspace.target_schema.w_dm_activity_type;

## Conversion Notes and Manual Actions Required

1.  **Surrogate Key Generation (`ACTIVITY_TYPE_SK`)**: The original ODI script used `ODI_SEQ_W_DM_ACTIVITY_TYPE.NEXTVAL` to generate `ACTIVITY_TYPE_SK`.
    In this conversion, `ABS(xxhash64(C_DM_ACTIVITY_TYPE.INTEGRATION_ID))` is used to generate a deterministic `BIGINT` surrogate key based on the `INTEGRATION_ID`.
    *   **Action**: Validate that `INTEGRATION_ID` is sufficiently unique for generating this key. If `INTEGRATION_ID` is not unique or a different strategy is required (e.g., a pure auto-incrementing identity column on the target table), this logic might need adjustment.
2.  **`ETL_PROC_WID` Parameter**: The `ETL_PROC_WID` is assumed to be a `BIGINT` (Oracle `NUMBER(20,0)`) and is passed as a widget. Ensure the calling process provides a valid integer value for this widget.
3.  **`WHERE (1=1)` Clause**: The redundant `WHERE (1=1)` clause in the staging insert has been removed.
4.  **`COMMIT;` Statement**: Oracle `COMMIT;` statements are implicit in Databricks Delta Lake transactions and have been removed.
5.  **Schema and Table Names**: Oracle schema names (`SOURCE_SCHEMA_DW`, `TARGET_SCHEMA_DW`) have been converted to `workspace.source_schema` and `workspace.target_schema` respectively. Staging table `C$_0W_DM_ACTIVITY_TYPE_0` has been renamed to `c_w_dm_activity_type_stg` following naming conventions.